# 1. Identity Fundamentals

## Identity is the new security perimeter

In the old world, your corporate network *was* the security boundary — everyone inside the firewall was trusted. That model is dead. People work from home, apps are in the cloud, and data flows everywhere.

**Identity** has replaced the network perimeter. Every request must prove *who* is asking (authentication) and *what* they're allowed to do (authorization).

## Setup

```bash
cd security/sc-900/02-identity-and-entra
docker compose up -d
uv sync
```

Select the `SC-900 (Python)` kernel.

---
## Authentication vs Authorization

These two get confused constantly. Here's the difference:

| | Authentication (AuthN) | Authorization (AuthZ) |
|-|----------------------|---------------------|
| **Question** | *Who are you?* | *What can you do?* |
| **Happens** | First | Second (after authn) |
| **Proof** | Password, MFA, biometric, certificate | Roles, permissions, policies |
| **Failure** | 401 Unauthorized | 403 Forbidden |
| **Azure service** | Entra ID | RBAC, Conditional Access |

**Analogy**: authentication is showing your passport at the airport. Authorization is whether your boarding pass lets you into first class.

In [ ]:
import httpx, json, base64

ENTRA = 'http://localhost:9000/contoso'
TOKEN_URL = f'{ENTRA}/oauth2/v2.0/token'
API_B = 'http://localhost:8002'

def decode(t):
    p = t.split('.')[1]
    return json.loads(base64.urlsafe_b64decode(p + '=' * (-len(p) % 4)))

# --- AUTHENTICATION: prove who you are ---
print('=== Step 1: Authentication (prove your identity) ===')
r = httpx.post(TOKEN_URL, data={
    'grant_type': 'password',
    'client_id': 'api-a-client-id',
    'client_secret': 'api-a-secret-value',
    'username': 'alice@contoso.com',
    'password': 'alice-password',
    'scope': 'api://api-b/Files.Read',
})
token = r.json()['access_token']
claims = decode(token)
print(f'✅ Authenticated as: {claims["upn"]}')
print(f'   Token audience: {claims["aud"]}')
print(f'   Scopes granted: {claims["scp"]}')

# --- AUTHORIZATION: check what you can do ---
print('\n=== Step 2: Authorization (can you access /files?) ===')
r = httpx.get(f'{API_B}/files', headers={'Authorization': f'Bearer {token}'})
print(f'Status: {r.status_code}')
print(json.dumps(r.json(), indent=2))

# --- Failed authentication ---
print('\n=== Step 3: Failed authentication (wrong password) ===')
r = httpx.post(TOKEN_URL, data={
    'grant_type': 'password',
    'client_id': 'api-a-client-id',
    'client_secret': 'api-a-secret-value',
    'username': 'alice@contoso.com',
    'password': 'WRONG-password',
    'scope': 'api://api-b/Files.Read',
})
print(f'Status: {r.status_code} — {r.json()["detail"]}')

# --- Failed authorization ---
print('\n=== Step 4: Failed authorization (no access to /files without token) ===')
r = httpx.get(f'{API_B}/files')
print(f'Status: {r.status_code} — {r.json()["detail"]}')

---
## Identity Providers (IdP)

An **identity provider** is the system that creates, maintains, and manages identity information and provides authentication services. Instead of every app managing its own usernames and passwords, you delegate to a central IdP.

| IdP | Who uses it |
|-----|-------------|
| **Microsoft Entra ID** | Azure, Microsoft 365, thousands of SaaS apps |
| Google Identity | Google Workspace, Android |
| Okta | Enterprise apps |
| On-premises AD | Traditional Windows domains |

### How it works (simplified)

```
User → App: "I want to log in"
App → IdP: "Please authenticate this user"
IdP → User: "Enter credentials + MFA"
User → IdP: credentials
IdP → App: signed token (JWT) proving user's identity
App: validates token signature, reads claims, grants access
```

The mock Entra server in this lab IS an identity provider — it issues signed JWTs just like real Entra ID.

In [ ]:
# The OIDC discovery document tells apps where to find the IdP's endpoints
discovery = httpx.get(f'{ENTRA}/v2.0/.well-known/openid-configuration').json()
print('OIDC Discovery Document (how apps find the IdP):\n')
for key in ['issuer', 'token_endpoint', 'jwks_uri']:
    print(f'  {key}: {discovery[key]}')

print('\n📖 In real Entra ID, this URL is:')
print('   https://login.microsoftonline.com/<tenant-id>/v2.0/.well-known/openid-configuration')

---
## Directory services and Active Directory

A **directory service** stores identity objects (users, groups, devices) in a hierarchical structure and provides lookup/authentication services.

| Service | Where | Protocol | Purpose |
|---------|-------|----------|---------|
| **Active Directory Domain Services (AD DS)** | On-premises | LDAP, Kerberos | Traditional Windows domain controller |
| **Microsoft Entra ID** | Cloud | OAuth2, OIDC, SAML | Cloud-native identity for Azure, M365, SaaS |
| **Microsoft Entra Domain Services** | Cloud | LDAP, Kerberos (managed) | Lift-and-shift AD to Azure without managing DCs |

### Exam tip

- AD DS = on-prem, you manage domain controllers
- Entra ID = cloud, no domain controllers, uses modern protocols (OAuth2/OIDC)
- Entra Domain Services = managed LDAP/Kerberos in Azure (for legacy apps)

They are **not** the same thing. AD DS ≠ Entra ID. Entra ID is not "AD in the cloud".

---
## Federation

**Federation** establishes trust between two identity systems so users from one can access resources in the other *without creating new accounts*.

Real-world example: Contoso (your company) partners with Fabrikam. Instead of creating Fabrikam accounts in your Entra tenant, you set up a **federated trust**. Fabrikam users authenticate with *their* IdP, and your apps trust the resulting token.

```
Fabrikam user → Fabrikam IdP: "authenticate me"
Fabrikam IdP → token (signed by Fabrikam)
Fabrikam user → Contoso app: "here's my Fabrikam token"
Contoso app → Contoso IdP: "is this token trustworthy?"
Contoso IdP: "yes, we trust Fabrikam's signing keys" → access granted
```

In Microsoft's world, **Entra External ID (B2B)** enables federation with other Entra tenants, Google, Facebook, or any SAML/OIDC IdP.

### Exam tip

Federation = trust relationship between IdPs. The user authenticates with their *home* IdP. No password syncing needed.

---
## Summary

| Concept | Key fact |
|---------|----------|
| **Authentication** | Proves *who* you are (401 if it fails) |
| **Authorization** | Determines *what* you can do (403 if it fails) |
| **Identity Provider** | Central system that authenticates users and issues tokens |
| **Directory service** | Stores users, groups, devices in a hierarchy |
| **AD DS** | On-prem, LDAP/Kerberos, you manage domain controllers |
| **Entra ID** | Cloud, OAuth2/OIDC, managed by Microsoft |
| **Federation** | Trust between IdPs — users don't need accounts in both |

**Next**: [Notebook 2 — Entra ID and Authentication](02_entra_id_and_authentication.ipynb)